# Environment Setup and Verification

This notebook verifies connectivity to the ADEPT ecosystem from JupyterLab.

## Security Architecture

JupyterLab is intentionally isolated on `frontend_net` — it has NO direct access to internal
services (Keycloak, orchestration, MCP servers, PostgreSQL, Redis). All API access routes
through `nginx_proxy` which enforces TLS and JWT authentication.

## Prerequisites
- ADEPT ecosystem services running (`make start`)
- JupyterLab container healthy (`docker ps | grep jupyterlab`)

## What This Notebook Does
1. Verifies Python environment
2. Checks connectivity to nginx_proxy
3. Acquires a JWT token and tests authenticated API access
4. Provides utility functions for other notebooks

## 1. Import Required Libraries

In [1]:
import os
import sys
import json
import httpx
import asyncio
from typing import Optional, Dict, Any
from datetime import datetime

# Add the src directory to Python path
sys.path.insert(0, '/app/src')

print(f"Python version: {sys.version}")
print(f"Current working directory: {os.getcwd()}")
print(f"Environment: {'Docker' if os.path.exists('/.dockerenv') else 'Host'}")

Python version: 3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 10:40:35) [GCC 12.3.0]
Current working directory: /home/jovyan/work/notebooks/00-setup
Environment: Docker


## 2. Environment Configuration

In [ ]:
# Service URLs - route through nginx_proxy for security isolation
# JupyterLab is on frontend_net only; all API access goes through nginx_proxy (TLS, JWT-gated)
if os.path.exists('/.dockerenv'):
    # Running inside Docker container — route through nginx reverse proxy
    NGINX_URL = "https://nginx_proxy"
    KEYCLOAK_TOKEN_URL = f"{NGINX_URL}/realms/master/protocol/openid-connect/token"
    KEYCLOAK_USERINFO_URL = f"{NGINX_URL}/realms/master/protocol/openid-connect/userinfo"
    API_BASE_URL = f"{NGINX_URL}/v1"
else:
    # Running on host machine
    NGINX_URL = "https://api.agentic.local"
    KEYCLOAK_TOKEN_URL = f"{NGINX_URL}/realms/master/protocol/openid-connect/token"
    KEYCLOAK_USERINFO_URL = f"{NGINX_URL}/realms/master/protocol/openid-connect/userinfo"
    API_BASE_URL = f"{NGINX_URL}/v1"

# Display configuration
print("Service Endpoints (all routed through nginx_proxy):")
print(f"  NGINX Proxy:     {NGINX_URL}")
print(f"  API Base URL:    {API_BASE_URL}")
print(f"  Keycloak Token:  {KEYCLOAK_TOKEN_URL}")
print(f"  Keycloak User:   {KEYCLOAK_USERINFO_URL}")
print()
print("Security model: JupyterLab is network-isolated on frontend_net.")
print("All service access is proxied through nginx (TLS + JWT validation).")

## 3. Service Health Checks

In [ ]:
async def check_service_health(name: str, url: str, path: str = "/health") -> Dict[str, Any]:
    """Check if a service is healthy via nginx_proxy."""
    try:
        async with httpx.AsyncClient(verify=False, timeout=5.0) as client:
            response = await client.get(f"{url}{path}")
            return {
                "service": name,
                "url": f"{url}{path}",
                "status": "HEALTHY" if response.status_code in [200, 204] else f"UNHEALTHY ({response.status_code})",
                "response_time": response.elapsed.total_seconds(),
            }
    except Exception as e:
        return {
            "service": name,
            "url": f"{url}{path}",
            "status": "UNREACHABLE",
            "error": str(e)
        }

# All health checks route through nginx_proxy
services = [
    ("NGINX Proxy", NGINX_URL, "/health"),
    ("Keycloak (via nginx)", NGINX_URL, "/realms/master/.well-known/openid-configuration"),
    ("Agent Gateway (via nginx)", NGINX_URL, "/v1/health"),
]

print("Checking service health via nginx_proxy...\n")
health_results = await asyncio.gather(*[
    check_service_health(name, url, path) for name, url, path in services
])

for result in health_results:
    status_indicator = "[OK]" if result["status"] == "HEALTHY" else "[FAIL]"
    print(f"  {status_indicator} {result['service']:30}")
    if 'response_time' in result:
        print(f"       URL: {result['url']} ({result['response_time']:.3f}s)")
    if 'error' in result:
        print(f"       Error: {result['error']}")
    print()

## 4. Install Required Python Packages

In [14]:
# Install required packages if not already installed
import subprocess
import importlib.util

required_packages = [
    "httpx",
    "openai",
    "python-jose",
    "python-keycloak",
    "pandas",
    "matplotlib",
    "seaborn",
    "scikit-learn"
]

missing_packages = []
for package in required_packages:
    if importlib.util.find_spec(package.replace("-", "_")) is None:
        missing_packages.append(package)

if missing_packages:
    print(f"Installing missing packages: {', '.join(missing_packages)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing_packages)
    print("✅ All packages installed successfully")
else:
    print("✅ All required packages are already installed")

Installing missing packages: python-jose, python-keycloak, scikit-learn
✅ All packages installed successfully


## 5. Verify API Access via nginx_proxy

MCP tool servers are internal (application_net) and not directly accessible from JupyterLab.
Tool invocation happens through the Agent Gateway API — send a chat message and the orchestration
service will route to the appropriate MCP server on your behalf.

In [ ]:
# Verify we can acquire a JWT token via nginx_proxy
import os

notebook_user = os.getenv("NOTEBOOK_USER_PASSWORD", "")
client_id = "validation-runner-client"

async def get_token():
    """Acquire JWT token via nginx_proxy Keycloak route."""
    async with httpx.AsyncClient(verify=False, timeout=10.0) as client:
        response = await client.post(
            KEYCLOAK_TOKEN_URL,
            data={
                "grant_type": "password",
                "client_id": client_id,
                "username": "notebook-user",
                "password": notebook_user,
            },
        )
        if response.status_code == 200:
            token_data = response.json()
            print("[OK] Token acquired via nginx_proxy")
            print(f"     Token type: {token_data.get('token_type')}")
            print(f"     Expires in: {token_data.get('expires_in')}s")
            return token_data.get("access_token")
        else:
            print(f"[FAIL] Token request failed: {response.status_code}")
            print(f"       Response: {response.text[:200]}")
            return None

token = await get_token()

if token:
    # Test authenticated API call
    async with httpx.AsyncClient(verify=False, timeout=10.0) as client:
        response = await client.get(
            f"{API_BASE_URL}/models",
            headers={"Authorization": f"Bearer {token}"},
        )
        if response.status_code == 200:
            models = response.json()
            print(f"\n[OK] Authenticated API call successful")
            print(f"     Available models: {len(models.get('data', []))}")
        else:
            print(f"\n[FAIL] API call failed: {response.status_code}")

## 6. Create Utility Functions

In [ ]:
# Utility functions for use in other notebooks

def get_api_base_url() -> str:
    """Get the API base URL (routes through nginx_proxy)."""
    return API_BASE_URL

def get_nginx_url() -> str:
    """Get the nginx_proxy URL."""
    return NGINX_URL

async def get_jwt_token(username: str = "notebook-user", password: str = None) -> Optional[str]:
    """Acquire a JWT token via nginx_proxy."""
    if password is None:
        password = os.getenv("NOTEBOOK_USER_PASSWORD", "")
    async with httpx.AsyncClient(verify=False, timeout=10.0) as client:
        response = await client.post(
            KEYCLOAK_TOKEN_URL,
            data={
                "grant_type": "password",
                "client_id": "validation-runner-client",
                "username": username,
                "password": password,
            },
        )
        if response.status_code == 200:
            return response.json().get("access_token")
        return None

async def api_request(method: str, path: str, token: str, **kwargs) -> Dict:
    """Make an authenticated API request via nginx_proxy."""
    async with httpx.AsyncClient(verify=False, timeout=30.0) as client:
        response = await client.request(
            method,
            f"{API_BASE_URL}{path}",
            headers={"Authorization": f"Bearer {token}"},
            **kwargs,
        )
        return {
            "status_code": response.status_code,
            "data": response.json() if response.text else None,
            "success": response.status_code < 400,
        }

# Save configuration for use in other notebooks
config = {
    "nginx_url": NGINX_URL,
    "api_base_url": API_BASE_URL,
    "keycloak_token_url": KEYCLOAK_TOKEN_URL,
    "environment": "docker" if os.path.exists('/.dockerenv') else "host",
    "timestamp": datetime.now().isoformat(),
}

config_file = "/tmp/adept_config.json"
with open(config_file, "w") as f:
    json.dump(config, f, indent=2)

print(f"Configuration saved to {config_file}")
print(json.dumps(config, indent=2))

## 7. Summary and Next Steps

In [ ]:
# Summary
print("=" * 60)
print("ENVIRONMENT SETUP COMPLETE")
print("=" * 60)

healthy_services = sum(1 for r in health_results if r["status"] == "HEALTHY")
total_services = len(health_results)

if healthy_services == total_services:
    print(f"[OK] All {total_services} services healthy via nginx_proxy")
    print()
    print("Next steps:")
    print("1. Continue to 01_authentication.ipynb for auth setup")
    print("2. Then explore notebooks based on your role:")
    print("   - End User: ../01-end-user/")
    print("   - Developer: ../02-developer/")
    print("   - Operator: ../03-operator/")
else:
    print(f"[WARN] {healthy_services}/{total_services} services healthy")
    print()
    print("Troubleshooting:")
    print("1. Ensure all services are started: make start")
    print("2. Check Docker containers: docker ps")
    print("3. Check nginx_proxy is running: docker logs nginx_proxy --tail 10")
    print("4. Verify JupyterLab is on frontend_net: docker inspect jupyterlab --format '{{json .NetworkSettings.Networks}}'")
    print()
    print("Architecture note:")
    print("  JupyterLab is isolated on frontend_net for security.")
    print("  All API access routes through nginx_proxy (TLS + JWT).")
    print("  Direct access to internal services is intentionally blocked.")